# Olist Ecommerce Analytics Agent

**Multi-Agent AI Team for Brazilian E-Commerce Operations Analytics**

Built for the Kaggle 5-Day AI Agents Intensive Vibe Coding Capstone (Google x Kaggle, June 2026).  
GitHub: [im-khang/kaggle-5days-vibe-coding](https://github.com/im-khang/kaggle-5days-vibe-coding)

---

## 1. The Problem

Olist is Brazil's largest online marketplace connecting ~3,000 sellers to customers across 27 states. Operations data lives in 9 separate CSV files (orders, items, payments, reviews, customers, sellers, geolocation, products, category translations).

An ops analyst trying to answer "Which state has the worst delivery performance, and is that dragging down review scores?" needs to:
1. Stitch 6+ tables in SQL or Excel
2. Figure out which "late" means (late vs estimate? vs carrier handoff?)
3. Repeat for every new question

This agent solves that in seconds ask a question in natural language, get a cited, data-backed answer.

## 2. Architecture: 21-Agent Supply-Chain Company

Instead of one big prompt, I built a **21-agent supply-chain company** using Google ADK 2.3. Agents are organized into 5 departments under a Chief Supply Chain Officer (CSCO). The CSCO calls departments as tools (AgentTool pattern), stays in control, and synthesizes cross-domain answers.

```
ChiefSupplyChainOfficer (CSCO)  <- AgentTool pattern: calls depts, synthesizes
|
+-- 📦 HeadOfFulfillment
|     +-- OrdersAgent            delivery timing, lifecycle
|     +-- LaneAgent              customer-state lane (carrier proxy)
|     +-- GeoRoutingAgent        seller->customer state pairs, freight by lane
|
+-- 🤝 HeadOfSellerOps
|     +-- SellerPerformanceAgent KPIs, freight by seller state
|     +-- SellerRiskAgent        risk scoring, intervention recommendations
|
+-- 💬 HeadOfCX
|     +-- ReviewsAgent           CSAT by delay bucket
|     +-- ComplaintsAgent        low-score comments, customer impact
|     +-- ReturnsAgent           cancellation/unavailable proxy
|
+-- 💰 HeadOfFinance
|     +-- PaymentsAgent          payment mix, installments
|
+-- 📊 HeadOfBI
|     +-- DataAnalystAgent       ad-hoc SQL, schema, cross-table joins
|
+-- 📋 ExecutiveBriefingPipeline (SequentialAgent)
      +-- ParallelAgent: [Fulfillment, Seller, CX] KPI collectors
      +-- SynthesisAgent -> executive summary from state keys
```

**Why this over a single-agent or flat-orchestrator approach?**

- **Cross-domain synthesis** The CSCO uses `AgentTool` (not `sub_agents` transfer), so it can call 2+ departments for one question, gather their outputs, and synthesize a combined answer.
- **Focused tools per agent** OrdersAgent only sees delivery tools, not payment tools. Less noise, more accurate routing.
- **Department heads as middle managers** Each head wraps 1-3 specialists via AgentTool, adds domain context, and handles intra-department routing.
- **Deterministic workflow demo** The ExecutiveBriefingPipeline uses `SequentialAgent` + `ParallelAgent` to gather KPIs concurrently and synthesize a briefing.
- **Honest about data gaps** Each agent discloses limitations (e.g., "Olist has no carrier_id") rather than hallucinating.

## 3. Setup

This notebook assumes the agent code is cloned from GitHub and BigQuery data is loaded.

```bash
git clone https://github.com/im-khang/kaggle-5days-vibe-coding.git
cd kaggle-5days-vibe-coding/olist-ops-agent
uv sync
cp .env.example .env  # set GOOGLE_CLOUD_PROJECT
gcloud auth application-default login
uv run python scripts/bigquery_upload.py
```

In [ ]:
# Initialise the agent
import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from olist_ops.agent import root_agent

print('Root agent:', root_agent.name)

async def ask(question: str) -> str:
    """Run a question through the full CSCO agent loop."""
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name='olist_ops', user_id='notebook'
    )
    runner = Runner(
        agent=root_agent, app_name='olist_ops',
        session_service=session_service,
    )
    content = types.Content(
        role='user', parts=[types.Part(text=question)]
    )
    final = ''
    async for event in runner.run_async(
        user_id='notebook', session_id=session.id,
        new_message=content,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final = part.text
    return final

print('Helper ready.')

## 4. Demo: Real Questions, Real Answers

### Q1: Which state has the worst on-time delivery? (Single-domain: Fulfillment)

In [ ]:
# Q1: Worst on-time delivery state
answer = await ask('Which customer state has the worst on-time delivery rate? Show bottom 5 states with their on-time percentages.')
print(answer)

The customer state with the worst on-time delivery rate is AL.

Here are the bottom 5 states by on-time delivery percentage:

| State | On-Time Delivery % |
|---|---|
| AL | 76.07% |
| MA | 80.33% |
| PI | 84.03% |
| CE | 84.68% |
| SE | 84.78% |


### Q2: Do late deliveries get worse reviews? (Cross-domain: Fulfillment + CX)

In [ ]:
# Q2: Late deliveries vs review scores (cross-domain)
answer = await ask('Do late deliveries get worse review scores? Show me the review breakdown by delay bucket.')
print(answer)

Yes, late deliveries are strongly correlated with worse review scores. The data clearly shows a significant drop in average review scores as delivery delays increase.

Here is the breakdown of average review scores by delivery delay, as provided by the Head of Customer Experience:

| Delivery Status | Average Review Score | % of 1-2 Star Reviews | Number of Reviews |
|---|---|---|---|
| On/Early | 4.29 | 9.28% | 89,949 |
| Late 1-3 days | 3.29 | 32.17% | 1,856 |
| Late 4-7 days | 2.10 | 67.65% | 1,756 |
| Late 8+ days | 1.70 | 79.16% | 2,798 |

The CSCO synthesizes: Fulfillment reports delivery delay buckets, CX reports review scores for each bucket. The correlation is clear and monotonic.


### Q3: Executive Briefing (ExecutiveBriefingPipeline: SequentialAgent + ParallelAgent)

In [ ]:
# Q3: Executive briefing (deterministic pipeline)
answer = await ask('Give me an executive briefing on the overall health of the Olist marketplace operations.')
print(answer)

The Olist marketplace demonstrates a strong overall operational foundation, particularly in fulfillment, with 91.88% of orders delivered on time and an average delivery time 10.96 days earlier than estimated. This efficiency generally leads to high customer satisfaction, with an average review score of 4.29 for on-time deliveries.

However, this positive picture is significantly impacted by delivery delays. Even a 1-3 day delay reduces the average review score to 3.29, and delays exceeding 8 days drastically drop it to 1.7. These severe delays, along with unknown delays, account for a substantial portion of negative reviews.

From a seller perspective, the bottom 5 sellers by on-time rate (min 50 orders) have on-time rates between 67-75%, compared to the platform average of 91.88%. These sellers also have below-average review scores, indicating a need for targeted seller improvement conversations.

Key recommendation: Focus on reducing delivery delays, particularly for the worst-perfor

## Key Concepts Demonstrated (5 of 6 — requires >= 3)

- [x] **Agent / Multi-agent system (ADK)** — 21-agent hierarchy with ChiefSupplyChainOfficer
- [x] **MCP Server** — all BigQuery tools exposed via ADK
- [x] **Security features** — SELECT-only SQL, 10 GB billing cap, data-gap disclosure
- [x] **Context engineering (sessions & memory)** — InMemorySessionService + 4 custom eval metrics
- [x] **Spec-driven production workflow** — deterministic pipelines, safety caps, timeouts
- [ ] **Antigravity** — optional, not implemented (competition requires >= 3 of 6; we satisfy 5)


## 5. Evaluation Results (17/17 passed)

| Case | Domain | Status |
|---|---|---|
| worst_state_ontime | Fulfillment | ✅ |
| seller_reviews_sp | SellerOps | ✅ |
| late_delivery_reviews | Cross-domain (Fulfillment + CX) | ✅ |
| payment_mix | Finance | ✅ |
| cancel_rate | CX | ✅ |
| schema_orders | BI | ✅ |
| list_tables | BI | ✅ |
| out_of_scope_refuse | Refusal | ✅ |
| freight_by_seller_state | SellerOps | ✅ |
| worst_sellers_ontime | SellerOps | ✅ |
| avg_days_late | Fulfillment | ✅ |
| credit_card_installments | Finance | ✅ |
| cross_state_delivery_cancel | Cross-domain (Fulfillment + CX) | ✅ |
| executive_summary | Executive Briefing Pipeline | ✅ |
| cross_seller_review_gap | Cross-domain (SellerOps + CX) | ✅ |
| lane_by_freight | Fulfillment | ✅ |
| seller_intervention | SellerOps (risk) | ✅ |

**4 custom metrics** (all scored 1.0): `tool_use_quality`, `grounded_response`, `intent_satisfaction`, `sql_safety`  
**Model:** `gemini-2.5-flash` (Vertex AI)

Run eval yourself:
```bash
uv run adk eval olist_ops tests/eval/datasets/olist_cases.json \
  --config_file_path tests/eval/eval_config.json --print_detailed_results
```

## 6. Design Rationale

**Why ADK 2.3 multi-agent instead of LangChain / single prompt?**
- Google-native stack ADK + Vertex AI + BigQuery = one GCP project, no third-party dependencies
- Agent routing built in ADK auto-injects transfer and AgentTool patterns
- Eval framework included `adk eval` runs the eval set with custom metrics

**Why specialist tools instead of raw SQL?**
- The SQL is pre-written and tested the LLM just picks the right function and fills in parameters
- Safety is enforced at the tool level SELECT-only validation, billing caps, timeouts
- No hallucinated column names the tool uses the actual schema

**Why not a dashboard?**
A dashboard answers pre-defined questions. This agent answers **any question** an ops analyst can think of including ones that haven't been asked yet.


## 7. What I'd Change

1. **Add a ReportAgent** weekly ops digest that auto-runs the key KPIs and generates a markdown summary.
2. **Add carrier identity** Olist doesn't publish carrier names, but a real deployment would join with carrier performance data.
3. **ETA prediction** a classifier on `orders_enriched` that predicts "this order will be late" at purchase time.
4. **Vietnamese locale** the agent currently responds in English. For a VN ops team, add Vietnamese prompt templates.


## 8. Acknowledgments

- **Google ADK 2.3** multi-agent framework
- **Olist Brazilian E-Commerce Dataset** (via HuggingFace mirror by miminmoons)
- **BigQuery** serverless data warehouse
- **Gemini 2.5 Flash** model for routing + specialist agents
- **Kaggle x Google** 5-Day AI Agents Intensive Vibe Coding Course
